In [1]:
from glob import glob
import numpy as np
import os
from malaya_speech.model.clustering import StreamingKMeans, StreamingSpeakerSimilarity
import malaya_speech
import numpy as np
from tqdm import tqdm
import torch
import torch.nn.functional as F
from typing import Callable


class StreamingSpeakerSimilarity:
    def __init__(self, similarity_threshold: float = 0.8, agg_function: Callable = torch.mean, device="cpu"):
        """
        Parameters
        ----------
        similarity_threshold : float, optional (default=0.8)
            If the current voice activity sample has similarity >= 0.8,
            we assume it is from the same speaker.
        agg_function : Callable, optional (default=torch.mean)
            Function to aggregate embeddings when updating a speaker.
        device : str, optional (default="cpu")
            Device to store embeddings ("cpu" or "cuda").
        """
        self.similarity_threshold = similarity_threshold
        self.agg_function = agg_function
        self.device = device

        self.speakers = {}          # dictionary {speaker_id: embedding}
        self.embeddings = None      # torch.Tensor [num_speakers, embedding_dim]

    def fit(self, data: torch.Tensor):
        """Fit with initial batch of samples."""
        for sample in data:
            self.streaming(sample)

    def streaming(self, sample: torch.Tensor) -> int:
        """Process one streaming sample."""
        sample = sample.to(self.device)

        if self.embeddings is not None:
            # Compute cosine similarity
            sim = F.cosine_similarity(sample.unsqueeze(0), self.embeddings)  # (num_speakers,)

            # Find candidates above threshold
            candidates = (sim >= self.similarity_threshold).nonzero(as_tuple=True)[0]

            if len(candidates) > 0:
                # Pick the most similar
                best_idx = candidates[sim[candidates].argmax().item()].item()
                new_embed = self.agg_function(
                    torch.stack([sample, self.speakers[best_idx]]), dim=0
                )
                self.speakers[best_idx] = new_embed
                self.embeddings[best_idx] = new_embed  # update in-place
                return best_idx

        # New speaker
        speaker = len(self.speakers)
        self.speakers[speaker] = sample

        if self.embeddings is None:
            self.embeddings = sample.unsqueeze(0)
        else:
            self.embeddings = torch.cat([self.embeddings, sample.unsqueeze(0)], dim=0)

        return speaker

class FastStreamingSpeakerSimilarity:
    def __init__(self, similarity_threshold=0.8, device="cpu"):
        self.similarity_threshold = similarity_threshold
        self.device = device
        self.embeddings = None  # shape: (num_speakers, D)

    def streaming(self, sample: torch.Tensor) -> int:
        # normalize sample
        sample = F.normalize(sample.unsqueeze(0).to(self.device), dim=-1)

        if self.embeddings is not None:
            # cosine similarity via matmul
            sim = torch.matmul(sample, self.embeddings.T).squeeze(0)  # (N,)

            candidates = (sim >= self.similarity_threshold).nonzero(as_tuple=True)[0]
            if len(candidates) > 0:
                best_idx = candidates[sim[candidates].argmax()].item()
                # update speaker embedding (moving average)
                self.embeddings[best_idx] = F.normalize(
                    (self.embeddings[best_idx] + sample.squeeze(0)) / 2, dim=-1
                )
                return best_idx

        # add new speaker
        if self.embeddings is None:
            self.embeddings = sample
        else:
            self.embeddings = torch.cat([self.embeddings, sample], dim=0)

        return self.embeddings.size(0) - 1

2025-09-26 06:00:40.804351: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758866440.813235  225916 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758866440.817421  225916 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758866440.822568  225916 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758866440.822579  225916 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758866440.822581  225916 computation_placer.cc:177] computation placer alr

In [2]:
files = glob('WenetSpeech4TTS_Premium/*.npy')
len(files)

252292

In [3]:
vectors = []
for f in tqdm(files):
    v = np.load(f)
    vectors.append(v)

100%|██████████| 252292/252292 [00:07<00:00, 32249.19it/s]


In [4]:
similarity = StreamingSpeakerSimilarity(device = 'cuda')

In [5]:
results_similarity = {}
for no, f in tqdm(enumerate(files)):
    v = vectors[no]
    i = int(f.split('/')[1].replace('.npy', ''))
    results_similarity[i] = similarity.streaming(torch.tensor(v))

252292it [07:20, 572.25it/s] 


In [6]:
len(results_similarity)

252292

In [8]:
len(set(results_similarity.values()))

74488